Зашружаем все необходимые библиотеки

In [5]:
import torch
from torch import nn 
import torch.nn.functional as F
import torch.optim.lr_scheduler as lr_sc
from torch.optim import Adam 
import numpy as np
from torch.utils.data import Dataset, random_split, DataLoader
import torchvision.io as io
from torchvision import transforms
import os
from pathlib import Path
import pandas as pd

In [6]:
classes = ['dog','automobile','deer','truck','frog'] 

Выбор классов обусловлен предположением, что тяжело разделимы 'dog' с 'deer' и 'truck' с 'automobile', а класс 'frog' будет наиболее часто определен верно

### Уменьшим набор данных до необходимого и изменим значения таргет на категориальные ,
 поскольку задача сведется к многоклассовой классификации а именно ф-ей потерь будет выбрана Кросс энтропия , то нет необходимости использовать one hot encoding

In [7]:
data_path = Path(r"C:\Users\Honor\Desktop\Data_arhive\cifar-10.0")
img_path = Path(r"C:\Users\Honor\Desktop\Data_arhive\cifar-10.0\train")

df = pd.read_csv(data_path /  "trainLabels.csv", delimiter=',')
target =  df[df['label'].isin(classes)].reset_index(drop = True)
target['num_label'] = pd.Categorical(target['label']).codes

In [8]:
dict_secret = {list(pd.unique(target['num_label']))[i]: list(pd.unique(target['label']))[i] for i in range(len(classes)) }
print(dict_secret)

{np.int8(3): 'frog', np.int8(4): 'truck', np.int8(1): 'deer', np.int8(0): 'automobile', np.int8(2): 'dog'}


In [9]:

# target['id'] = (target['id']).astype(str) + ".pdf"
image = sorted(os.listdir(img_path), key = lambda x: int(Path(x).stem))
# print([((image[x]).split("."))[0] for x in range(31)] )

# for f in img_path.glob("*png"):
#     if (int(f.stem)) not in target['id'].values:
#             f.unlink()
label =  target['label'].reset_index(drop = True)
data = (image, list(label))


Настроим класс загрузки и предобработки данных 

In [6]:
# classes = ['dog','automobile','deer','truck','frog']

# class My_DataLoader(Dataset):
#     def __init__(self, path):
#         self.path = path
#         self.image = sorted(os.listdir(path / "train"), key= lambda x : int(Path(x).stem))
#         self.target = target['label'].reset_index(drop = True)
#         self.transform =  transforms.TrivialAugmentWide()
        
#     def __len__(self):
#         return len(self.target)
    
#     def __getitem__(self,idx):
#         img_path = self.image / "test" /self.image[idx]
#         img_tensor = io.read_image(str(img_path)) # -> [C, H, W]
#         if self.transform:
#             img_tensor = self.transform(img_tensor)
#         label = self.target
#         return img_tensor, label
              
# data_path = Path(r"C:\Users\Honor\Desktop\Data_arhive\cifar-10.0")
# data = My_DataLoader(data_path)   # C:\Users\Honor\Desktop\Data_arhive\cifar-10.0


In [10]:
class My_T_DataSET(Dataset):
    def __init__(self, data):
        # self.subset = subset
        self.image = data[0]
        self.target = data[1]
        
    def __len__(self):
        return len(self.target)
    
    def __getitem__(self,idx):
        img_name = self.image[idx]
        img_tensor = io.read_image(str(img_path / img_name)) # -> [C, H, W]
        label = torch.tensor(self.target[idx])
        return img_tensor, label
    
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img.float()/255, label.float()

Выбор аугментации (TrivialAugmentWide) обусловлен выводами статьи 
"TrivialAugment: Tuning-free Yet State-of-the-Art Data Augmentation" (ICCV 2021),
которые говорят о том, что применение TrivialAugment
показывает или более высокие показатели или не хуже чем вычислительно затратные методы: AutoAugment(который включал подбор наилучшей аугментации из десятков заданных) -> RandAugment (которая заключалась в выборе слцчайных N аушментаций и подборе силы - M параметровпреобразования ). Следующей ступенью стало отказаться от какого либо подбора настроек и выбору всего ОДНОГО преобразования на изображение, что и привело к TrivialAugment. 
Данный способ позволил обучать модель инвариантностям.

Разделим данные на тренировочные валидационные и тестовые 

In [11]:
len(data[0])

25000

In [12]:
n = len(image)
train_data = int(0.75*n)
val_data = int(0.15*n)
test_data = n - (train_data + val_data)

data = My_T_DataSET(data)
train_data_ ,val_data_, test_data_ = random_split(data, [train_data,val_data,test_data])
train_l = TransformSubset(train_data_, transforms.TrivialAugmentWide())
val_l = TransformSubset(val_data_)
test_l = TransformSubset(test_data_)

train_loader = DataLoader(train_l, batch_size= 64, shuffle=True)
val_loader = DataLoader(val_l, batch_size= 64)
test_loader = DataLoader(test_l,batch_size= 64)


In [16]:
type(test_loader)

torch.utils.data.dataloader.DataLoader

In [15]:
images , labels = next(iter(train_loader))

print(f"Shape of X [N, C, H, W]: {images.shape}, {images.dtype}")
print(f"Shape of y: {labels.shape}, {labels.dtype}")

TypeError: new(): invalid data type 'str'

In [11]:
import matplotlib.pyplot as plt

titles = [dict_secret[y] for y in yy][0:14]
fig, axes = plt.subplots(2,7, figsize=(15, 6))
i = 0
for axs, y in zip(axes.flatten(), titles):
    axs.imshow(XX[i].permute(1, 2, 0)) 
    axs.set_title(y)
    axs.axis("off")
    i+=1

plt.tight_layout()
plt.show()


TypeError: 'int' object is not iterable

Создание базовой архитектуры CNN для классификации 